# Lesson 9: Transfer learning and evaluation

Goes with the Lesson 9 page of the Forkcast course. A pretrained ResNet-18, frozen; a new head trained on cached features in seconds; the top block fine-tuned with its own learning rate; a confusion matrix to read.

Runtime: pick a **GPU**. The pretrained weights (45 MB) download once.

## 1. Clone your repository and install the package

In [ ]:
GITHUB_USER = "your-github-username"   # edit this line
REPO = "forkcast-course"

!git clone https://github.com/{GITHUB_USER}/{REPO}.git
%cd {REPO}
!pip install -e .

## 2. Twenty dishes at 224×224

Streamed from the Hugging Face Hub: 150 training and 50 validation photos per class, preprocessed exactly as the pretrained network expects.

In [ ]:
!pip install -q datasets
import importlib, torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from torchvision import models
from torchvision.utils import make_grid
from datasets import load_dataset
from forkcast import train, vision
from forkcast.model import count_parameters

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = models.ResNet18_Weights.DEFAULT
prep = weights.transforms()                              # resize 256, crop 224, ImageNet normalization
print(prep)

KEEP = list(range(0, 101, 5))[:20]                       # twenty of the 101 classes
stream = load_dataset("ethz/food101", split="train", streaming=True)
names = [stream.features["label"].names[k] for k in KEEP]
per_class = {k: 0 for k in KEEP}; xs, ys = [], []
for ex in stream:
    k = ex["label"]
    if k in per_class and per_class[k] < 200:
        xs.append(prep(ex["image"].convert("RGB"))); ys.append(KEEP.index(k)); per_class[k] += 1
    if all(v >= 200 for v in per_class.values()):
        break
X = torch.stack(xs); Y = torch.tensor(ys)
perm = torch.randperm(len(Y), generator=torch.Generator().manual_seed(0))
X, Y = X[perm], Y[perm]
n_val = 20 * 50
train_loader = DataLoader(TensorDataset(X[n_val:], Y[n_val:]), batch_size=64, shuffle=True, drop_last=True)
val_loader = DataLoader(TensorDataset(X[:n_val], Y[:n_val]), batch_size=64)
print(X.shape, names)

## 3. `forkcast/transfer.py`

Write `freeze`, `build_transfer_model`, `param_groups`, `top_k_accuracy` and `confusion_matrix`; the feature cache and the matrix helpers are provided.

In [ ]:
%%writefile forkcast/transfer.py
"""Lesson 9. Transfer learning: start from a network that already sees, and honest evaluation.

``extract_features``, ``per_class_accuracy`` and ``most_confused`` are provided; the rest are exercises.
"""

from __future__ import annotations

import torch
from torch import Tensor, nn
from torchvision import models

BACKBONE_BLOCKS = ["layer4", "layer3", "layer2", "layer1"]   # from the top of the network downwards


def freeze(module: nn.Module) -> nn.Module:
    """Switch off gradients for every parameter in ``module`` and return it."""
    raise NotImplementedError("Lesson 9, step 3: for p in module.parameters(): p.requires_grad_(False)")


def build_transfer_model(n_classes: int, pretrained: bool = True, trainable_blocks: int = 0) -> nn.Module:
    """A ResNet-18 that classifies our dishes.

    1. ``models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)``
    2. freeze the whole network
    3. replace ``model.fc`` with a fresh ``nn.Linear(model.fc.in_features, n_classes)`` (new layers require grad by default)
    4. for the first ``trainable_blocks`` entries of BACKBONE_BLOCKS, switch gradients back on with ``requires_grad_(True)``
    """
    raise NotImplementedError("Lesson 9, step 3: the four steps in the docstring")


def param_groups(model: nn.Module, lr_head: float, lr_backbone: float) -> list[dict]:
    """Two groups for the optimizer: the head (``model.fc``) at ``lr_head``, every other trainable parameter at ``lr_backbone``.

    Return ``[{"params": [...], "lr": lr_head}, {"params": [...], "lr": lr_backbone}]``, skipping frozen parameters.
    """
    raise NotImplementedError("Lesson 9, step 5: split model.named_parameters() by whether the name starts with \"fc.\"")


def top_k_accuracy(logits: Tensor, y: Tensor, k: int = 5) -> float:
    """Fraction of rows whose true class is among the k largest logits. (N, C), (N,) -> float."""
    raise NotImplementedError("Lesson 9, step 6: logits.topk(k, dim=1).indices == y[:, None], any over dim 1, mean")


def confusion_matrix(preds: Tensor, y: Tensor, n_classes: int) -> Tensor:
    """(N,), (N,) -> (n_classes, n_classes) int64: rows are true classes, columns are predictions.

    cm[i, j] counts images of class i predicted as class j. A loop over the pairs is fine;
    ``torch.bincount(y * n_classes + preds, minlength=n_classes ** 2).view(n_classes, n_classes)`` is the one-liner.
    """
    raise NotImplementedError("Lesson 9, step 6: the bincount one-liner, or a loop")


@torch.no_grad()
def extract_features(model: nn.Module, loader, device: str | torch.device = "cpu") -> tuple[Tensor, Tensor]:
    """Run the frozen backbone once over a loader and cache its 512-d features: the fast way to train a head."""
    model.eval().to(device)
    fc, model.fc = model.fc, nn.Identity()            # temporarily return the pooled features instead of logits
    feats, labels = [], []
    for x, y in loader:
        feats.append(model(x.to(device)).cpu())
        labels.append(y)
    model.fc = fc
    return torch.cat(feats), torch.cat(labels)


def per_class_accuracy(cm: Tensor) -> Tensor:
    """Diagonal over row sums: how often each true class is recognised."""
    return cm.diagonal().float() / cm.sum(dim=1).clamp_min(1).float()


def most_confused(cm: Tensor, k: int = 5) -> list[tuple[int, int, int]]:
    """The k largest off-diagonal entries as (true, predicted, count)."""
    off = cm.clone()
    off.fill_diagonal_(0)
    flat = off.flatten()
    idx = flat.argsort(descending=True)[:k]
    n = cm.shape[0]
    return [(int(i // n), int(i % n), int(flat[i])) for i in idx if flat[i] > 0]

### Freeze, replace the head

In [ ]:
from forkcast import transfer
importlib.reload(transfer)
model = transfer.build_transfer_model(20)                # pretrained=True: downloads the weights once
print("trainable:", count_parameters(model), "of", sum(p.numel() for p in model.parameters()))
x_batch, _ = next(iter(val_loader))
print(model.to(device)(x_batch.to(device)).shape)

## 4. Cache the features, train the head in seconds

In [ ]:
feats, ys_tr = transfer.extract_features(model, train_loader, device)
vfeats, ys_val = transfer.extract_features(model, val_loader, device)
print(feats.shape, vfeats.shape)
head_train = DataLoader(TensorDataset(feats, ys_tr), batch_size=128, shuffle=True)
head_val = DataLoader(TensorDataset(vfeats, ys_val), batch_size=256)
torch.manual_seed(0)
head = nn.Linear(512, 20)
h = train.fit_v2(head, head_train, head_val, epochs=30, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=device)
model.fc.load_state_dict(head.state_dict())             # the full model now classifies pixels
print("head on frozen features: validation accuracy", round(vision.accuracy_loader(model, val_loader, device), 3))
print("Lesson 7 from scratch on photos like these was far lower; chance is 0.05")

## 5. Fine-tune layer4 with two learning rates

In [ ]:
importlib.reload(transfer)
ft = transfer.build_transfer_model(20, trainable_blocks=1).to(device)
ft.fc.load_state_dict(head.state_dict())
print("trainable:", count_parameters(ft))
optimizer = torch.optim.Adam(transfer.param_groups(ft, lr_head=1e-3, lr_backbone=1e-4))
scaler = train.make_scaler(device, amp=True)
for epoch in range(3):
    loss = train.train_one_epoch_amp(ft, train_loader, optimizer, nn.CrossEntropyLoss(), device, amp=True, scaler=scaler)
    print(f"epoch {epoch}: train loss {loss:.3f}, validation accuracy {vision.accuracy_loader(ft, val_loader, device):.3f}")

## 6. Evaluate honestly

In [ ]:
@torch.no_grad()
def collect_logits(m, loader):
    m.eval(); out, ys_ = [], []
    for x, y in loader:
        out.append(m(x.to(device)).cpu()); ys_.append(y)
    return torch.cat(out), torch.cat(ys_)

logits, y = collect_logits(ft, val_loader)
print("top-1", round(transfer.top_k_accuracy(logits, y, 1), 3), "top-5", round(transfer.top_k_accuracy(logits, y, 5), 3))
cm = transfer.confusion_matrix(logits.argmax(1), y, 20)
acc = transfer.per_class_accuracy(cm)
for i in acc.argsort()[:5]:
    print(f"hardest: {names[i]:22s} {acc[i]:.2f}")
print("most confused:", [(names[i], names[j], n) for i, j, n in transfer.most_confused(cm, 5)])
plt.figure(figsize=(7, 6)); plt.imshow(cm, cmap="Blues"); plt.xticks(range(20), names, rotation=90, fontsize=7); plt.yticks(range(20), names, fontsize=7); plt.xlabel("predicted"); plt.ylabel("true");

### Look at the worst pair

In [ ]:
i, j, n = transfer.most_confused(cm, 1)[0]
wrong = (y == i) & (logits.argmax(1) == j)
imgs = X[:n_val][wrong][:8]
mean = torch.tensor(prep.mean)[:, None, None]; std = torch.tensor(prep.std)[:, None, None]
plt.figure(figsize=(12, 2)); plt.axis("off"); plt.title(f"{names[i]} predicted as {names[j]}")
plt.imshow(make_grid((imgs * std + mean).clamp(0, 1), nrow=8).permute(1, 2, 0))

### Break it on purpose: everything trainable, one big learning rate

In [ ]:
reckless = transfer.build_transfer_model(20, trainable_blocks=4).to(device)
h = train.fit_v2(reckless, train_loader, val_loader, epochs=2, lr=1e-3, loss_fn=nn.CrossEntropyLoss(), device=device, amp=True)
print("all 11M parameters at lr=1e-3: validation accuracy", round(vision.accuracy_loader(reckless, val_loader, device), 3), "(the frozen head alone did far better)")

## 7. Definition of done

Expected: `7 passed`, then `69 passed` with no failures for everything so far (skipped tests belong to lessons you have not started).

In [ ]:
!pytest tests/test_09.py -q

In [ ]:
!pytest -m fast -q

## 8. Commit and push

In [ ]:
from getpass import getpass
TOKEN = getpass("GitHub token with access to your repo: ")

!git config user.name "{GITHUB_USER}"
!git config user.email "{GITHUB_USER}@users.noreply.github.com"
!git add -A
!git commit -m "Lesson 9: transfer learning and evaluation"
!git push https://{TOKEN}@github.com/{GITHUB_USER}/{REPO}.git HEAD:main